# Exploration: stylised facts of daily returns

Run `make data` first (or set `CONFIG = 'config/synthetic.yaml'`). This notebook only *looks* at the data; every computation used in results lives in `src/voltail`.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from voltail.config import load_config
from voltail.data.store import load_processed
CONFIG = None  # or 'config/synthetic.yaml'
cfg = load_config(CONFIG)
data = load_processed(cfg)
list(data)

## 1. Volatility clustering: returns are ~uncorrelated, squared returns are not

In [ ]:
def acf(x, lags=50):
    x = (x - x.mean()) / x.std()
    return [np.mean(x[k:] * x[:-k]) for k in range(1, lags + 1)]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for a, df in data.items():
    r = df['ret'].to_numpy()
    axes[0].plot(acf(r), lw=1, label=a)
    axes[1].plot(acf(r**2), lw=1, label=a)
axes[0].set_title('ACF of returns'); axes[1].set_title('ACF of squared returns'); axes[1].legend(ncol=2)
for ax in axes: ax.axhline(0, color='k', lw=0.6)

## 2. Fat tails: compare return quantiles with a normal distribution

In [ ]:
rows = []
for a, df in data.items():
    r = df['ret']
    rows.append({'asset': a, 'ann_vol_%': r.std() * np.sqrt(252), 'skew': stats.skew(r), 'excess_kurtosis': stats.kurtosis(r),
                 'worst_day_%': r.min(), 'days_below_-4sd': int((r < r.mean() - 4 * r.std()).sum()),
                 'expected_if_normal': len(r) * stats.norm.cdf(-4)})
pd.DataFrame(rows).round(3)

## 3. Leverage effect: do down days raise next-day range more than up days?

In [ ]:
for a, df in data.items():
    nxt = np.log(df['garman_klass'].shift(-1))
    q = pd.qcut(df['ret'], 10, labels=False)
    plt.plot(nxt.groupby(q).mean().to_numpy(), marker='o', lw=1, label=a)
plt.xlabel('decile of today\'s return (0 = worst)'); plt.ylabel('mean log GK tomorrow'); plt.legend(ncol=2)